In [ ]:
from pathlib import Path
import json
import IPython.display as ipd

import torch

from hifigan.models import Generator as HiFiGAN
from hifigan.utils import AttrDict

In [ ]:
# load hifigan
device = "cpu"
hifigan_dir = Path("/Users/cafr02/repos/spkanon/checkpoints/knnvc")
config = json.load(open(hifigan_dir/"hifigan.json"))
h = AttrDict(config)
hifigan = HiFiGAN(h).to(device)

hifigan_ckpt = torch.load(
    "/Users/cafr02/repos/spkanon/hifigan_logs/g_00120000.pt", map_location="cpu"
)
# hifigan_ckpt = torch.load(hifigan_dir/"hifigan.pt", map_location=device)
hifigan.load_state_dict(hifigan_ckpt["generator"])
hifigan.eval()
hifigan.remove_weight_norm()

In [ ]:
prematch_dir = Path("ls_test_clean_prematch")
n_spk = 3
n_utts_per_spk = 1

spk_dirs = sorted([p for p in prematch_dir.iterdir() if p.is_dir()])[:n_spk]

for spk_dir in spk_dirs:
    pt_files = sorted(spk_dir.glob("*/*.pt"))[:n_utts_per_spk]
    for pt in pt_files:
        
        print(pt)
        feats = torch.load(pt).float().unsqueeze(0)
        with torch.inference_mode():
            out_wav = hifigan(feats)

        ipd.display(ipd.Audio(out_wav.squeeze().numpy(), rate=16000))